In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis Resume (`emcee`): LBCO, HRPT

This tutorial shows how to reopen the Bayesian project created previously,
inspect the saved fit results and then run more sampling steps to
extend the existing chain. Resuming only works with EMCEE because the
current BUMPS-DREAM implementation does not support saving and
resuming its state.

This workflow is useful when:
- the initial sampling run has not yet converged and more steps are needed,
- the initial sampling run has converged but more steps are desired
  for better posterior resolution,
- the initial sampling run has converged but the posterior plots have
  not yet been inspected and the user wants to see the plots before
  deciding whether to run more steps.

The workflow uses the same La0.5Ba0.5CoO3 powder diffraction example
as the DREAM Bayesian tutorial:

- run a short local refinement,
- derive finite fit bounds for the sampled parameters,
- switch to emcee and sample the posterior,
- save the project with the emcee chain,
- resume the chain with additional steps,
- inspect posterior plots after each sampling stage.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📂 Load Project

### Locate Project

Download and extract the saved emcee project, with the persisted chain
and posterior caches, from the EasyDiffraction data repository.

In [3]:
project_dir = edi.download_data('proj-lbco-hrpt-emcee', destination='projects')

Getting data...


Data 'proj-lbco-hrpt-emcee': Bayesian Analysis (emcee): LBCO, HRPT


✅ Data 'proj-lbco-hrpt-emcee' downloaded and extracted to '../../../projects/proj-lbco-hrpt-emcee-872a777e3a48'


### Load Project

Loading restores the persisted fit state, posterior samples, and plot
caches. No new fit is launched in this tutorial.

In [4]:
project = edi.Project.load(project_dir)

⚠️ Switching minimizer type removes these settings:                                                                               
   • max_iterations                                                                                                               


⚠️ Switching minimizer type adds these settings with defaults:                                                                    
   • burn_in_steps=1000                                                                                                           
   • initialization_method='ball'                                                                                                 
   • parallel_workers=0                                                                                                           
   • population_size=32                                                                                                           
   • proposal_moves='de'                                                                                                          
   • random_seed=None                                                                                                             
   • sampling_steps=5000                                                           

Re-save the project to a fresh working directory so resuming the
chain below writes there instead of the bundled read-only copy.

In [5]:
project.save_as(dir_path='projects/bayesian-emcee-resume-lbco-hrpt')

Saving project 📦 'lbco_hrpt_emcee' to '../../../projects/bayesian-emcee-resume-lbco-hrpt'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 results.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_emcee.html


## 📊 Inspect Results

### Display Structure

Render the La0.5Ba0.5CoO3 structure restored from the saved project.

In [6]:
project.display.structure(struct_name='lbco')

Structure 🧩 'lbco' (Atom view type: 'covalent')


### Display Fit Results

The fit summary reports the committed point estimate, sampler
settings, convergence diagnostics, and posterior parameter summaries
from the saved Bayesian run.

In [7]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,sampling_steps,100,Total sampler iterations per chain.
2,burn_in_steps,20,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,16,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,ball,emcee walker initialization method.
7,random_seed,42,Random seed; None uses a system-derived seed.
8,proposal_moves,de,Single emcee proposal move; move mixtures are not persisted in v1.


📋 Bayesian fit results:


,Metric,Value
1,🧪 Sampler,emcee
2,❌ Overall status,failed
3,💬 Engine message,emcee sampling completed
4,⏱️ Fitting time (seconds),80.66
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.65
7,"📏 R-factor squared (Rf², %)",4.92
8,"📏 Weighted R-factor (wR, %)",4.08
9,📉 Best log-posterior,-1157.01
10,📊 Convergence status,failed


📈 Committed parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↑
2,hrpt,linked_structure,lbco,scale,,9.1330,9.1290,0.0290,0.04 % ↓
3,hrpt,peak,,broad_gauss_u,deg²,0.0816,0.0821,0.0060,0.62 % ↑
4,hrpt,peak,,broad_gauss_v,deg²,-0.1169,-0.1171,0.0043,0.21 % ↑
5,hrpt,instrument,,twotheta_offset,deg,0.6303,0.6303,0.0014,0.00 % ↑


📊 Posterior distribution:


,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.225,166.1
2,hrpt,linked_structure,lbco,scale,,9.1324,"[9.0816, 9.1860]",1.182,188.0
3,hrpt,peak,,broad_gauss_u,deg²,0.0818,"[0.0706, 0.0933]",1.270,183.4
4,hrpt,peak,,broad_gauss_v,deg²,-0.1168,"[-0.1256, -0.1088]",1.262,195.7
5,hrpt,instrument,,twotheta_offset,deg,0.6302,"[0.6272, 0.6333]",1.249,159.6


### Display Correlations

The correlation matrix is restored from the saved project state.

In [8]:
project.display.fit.correlations()

### Display Posterior Densities

The pair plot and one-dimensional posterior distributions now load
from the persisted caches generated when the Bayesian fit was saved.

In [9]:
project.display.posterior.pairs()

In [10]:
project.display.posterior.distribution()

### Display Posterior Predictive

The posterior predictive view reuses the cached predictive summary
stored in the project rather than recalculating it on first display.
It overlays the 95% credible interval propagated from the posterior
samples.

In [11]:
project.display.posterior.predictive(expt_name='hrpt')

A zoomed view is useful for checking the propagated uncertainty in a
narrow region of the diffraction pattern.

In [12]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)

## 🎲 Resume Sampling

### Run Sampling

Resume from the saved backend and append 100 more emcee steps to the
existing chain. We use only 100 steps here to keep the tutorial fast,
but in practice you would typically run more steps to ensure
convergence and better posterior resolution.

In [13]:
project.analysis.minimizer.random_seed = 42  # fixed seed for reproducible output
project.analysis.fit(resume=True, extra_steps=100)

⚠️ resume=True requested, but no saved emcee chain was found; starting a fresh fit instead.                                       


<IPython.core.display.Javascript object>

⚠️ Existing fit results sidecar                                                                                                   
   '/home/runner/work/diffraction-lib/diffraction-lib/projects/bayesian-emcee-resume-lbco-hrpt/analysis/results.h5' will be       
   overwritten when the new fit is saved.                                                                                         


Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 results.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_emcee.html


In [14]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,sampling_steps,100,Total sampler iterations per chain.
2,burn_in_steps,20,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,16,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,ball,emcee walker initialization method.
7,random_seed,42,Random seed; None uses a system-derived seed.
8,proposal_moves,de,Single emcee proposal move; move mixtures are not persisted in v1.


📋 Bayesian fit results:


,Metric,Value
1,🧪 Sampler,emcee
2,❌ Overall status,failed
3,💬 Engine message,emcee sampling completed
4,⏱️ Fitting time (seconds),96.69
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.65
7,"📏 R-factor squared (Rf², %)",4.92
8,"📏 Weighted R-factor (wR, %)",4.09
9,📉 Best log-posterior,-1157.03
10,📊 Convergence status,failed


📈 Committed parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↑
2,hrpt,linked_structure,lbco,scale,,9.1290,9.1316,0.0288,0.03 % ↑
3,hrpt,peak,,broad_gauss_u,deg²,0.0821,0.0819,0.0063,0.24 % ↓
4,hrpt,peak,,broad_gauss_v,deg²,-0.1171,-0.1169,0.0043,0.14 % ↓
5,hrpt,instrument,,twotheta_offset,deg,0.6303,0.6303,0.0017,0.00 % ↓


📊 Posterior distribution:


,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.191,152.8
2,hrpt,linked_structure,lbco,scale,,9.1348,"[9.0828, 9.2085]",1.238,173.0
3,hrpt,peak,,broad_gauss_u,deg²,0.0811,"[0.0681, 0.0920]",1.174,177.8
4,hrpt,peak,,broad_gauss_v,deg²,-0.1163,"[-0.1232, -0.1074]",1.185,186.7
5,hrpt,instrument,,twotheta_offset,deg,0.6305,"[0.6265, 0.6335]",1.202,163.3


### Display Resumed Posterior

After resume, the posterior plots use the extended chain.

In [15]:
project.display.posterior.pairs()

In [16]:
project.display.posterior.distribution()

In [17]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)

## 💾 Save Project

In [18]:
project.save_as(dir_path='projects/bayesian-emcee-resume-lbco-hrpt')

Saving project 📦 'lbco_hrpt_emcee' to '../../../projects/bayesian-emcee-resume-lbco-hrpt'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 results.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_emcee.html
